# Solutions · 03 · Shear-thinning fluids

Worked solutions to the exercises of [notebook 03](../notebooks/03_shear_thinning_fluids.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import datasets, fitting, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
df = pd.read_csv(datasets.path("xanthan_flow_curve.csv"), comment="#")
rate, tau = df.shear_rate_1_s.to_numpy(), df.shear_stress_Pa.to_numpy()
best = fitting.fit_flow_curve(rate, tau, "carreau")

## Exercise 1

Fit a power law to the data between 10 and 1000 1/s only. By what factor does it overestimate the viscosity at 0.01 1/s? How does that factor change if you fit from 1 to 1000 1/s instead?

In [3]:
for lo in (10.0, 1.0):
    m = rate >= lo
    pl = fitting.fit_flow_curve(rate[m], tau[m], "power_law")
    print(f"power law fitted from {lo:g} 1/s: n = {pl.params['n']:.3f}; predicted/measured viscosity at 0.01 1/s: "
          f"{pl.predict_viscosity(0.01) / (tau[0]/rate[0]):.1f}")

power law fitted from 10 1/s: n = 0.262; predicted/measured viscosity at 0.01 1/s: 4.6
power law fitted from 1 1/s: n = 0.258; predicted/measured viscosity at 0.01 1/s: 4.8


Fitting from 1 1/s instead of 10 1/s changes little: the power law still overestimates the viscosity at rest
almost fivefold. No choice of range can make a power law describe a plateau; a model with a zero-shear
viscosity is needed.

## Exercise 2

Pulp particles (diameter 0.3 mm, 50 kg/m³ denser than the drink) settle under gravity. Estimate the Stokes settling velocity $v = \Delta\rho\, g\, d^2/(18\eta)$ in water and in the xanthan drink (using the viscosity at rest). Check that the shear rate $v/d$ around the particle really lies on the viscosity plateau.

In [4]:
d, drho, g = 0.3e-3, 50.0, 9.81
for name, eta in (("water", 1.0e-3), ("xanthan drink (at rest)", best.params["eta0"])):
    v = drho * g * d**2 / (18 * eta)
    print(f"{name:<24}: settling velocity {v:.2e} m/s = {v*3.6e6:.3g} mm/h, particle shear rate v/d = {v/d:.1e} 1/s")
print(f"onset of shear thinning 1/lambda = {1/best.params['lam']:.3f} 1/s")

water                   : settling velocity 2.45e-03 m/s = 8.83e+03 mm/h, particle shear rate v/d = 8.2e+00 1/s
xanthan drink (at rest) : settling velocity 7.03e-08 m/s = 0.253 mm/h, particle shear rate v/d = 2.3e-04 1/s
onset of shear thinning 1/lambda = 0.083 1/s


In water the pulp would sink by about 9 m per hour; in the drink by a fraction of a millimetre per hour -
the shelf-life problem is solved. The shear rate around a settling particle is far below the onset of shear
thinning, so the plateau viscosity is the right one to use - and a power-law model, whose viscosity has no
plateau, would have predicted nonsense here.

## Exercise 3

**Implement it yourself:** compute AIC $= n\ln(\text{SSR}/n) + 2k$ from the log residuals of the Carreau fit and compare with `best.aic`; then do the same for the power-law fit.

In [5]:
for f in (best, fitting.fit_flow_curve(rate, tau, "power_law")):
    ssr = np.sum(f.log_residuals**2)
    aic = f.n * np.log(ssr / f.n) + 2 * len(f.params)
    print(f"{f.model:<10}: AIC by hand {aic:.3f}, library {f.aic:.3f}")

carreau   : AIC by hand -248.914, library -248.914
power_law : AIC by hand -70.770, library -70.770


AIC $= n\ln(\text{SSR}/n) + 2k$: the fit term rewards small residuals, the penalty term $2k$ charges for each
parameter. The Carreau model's two extra parameters cost 4 AIC units but reduce the residuals enormously.